In [17]:
from pathlib import Path
import shutil
import yaml

SRC = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug")
DST = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug_v2")

if DST.exists():
    shutil.rmtree(DST)
    print("Deleted old archive4_yolo_aug_v2")

shutil.copytree(SRC, DST)
print("Copied dataset to:", DST)

# Remove old YOLO cache files, because copied cache can point to old paths
for cache_file in DST.rglob("*.cache"):
    cache_file.unlink()
    print("Removed cache:", cache_file)

# Fix data.yaml to point to v2 folder
yaml_path = DST / "data.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

data["path"] = DST.as_posix()
data["train"] = "images/train"
data["val"] = "images/val"

with open(yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("\nUpdated data.yaml:")
print(yaml_path.read_text())

Copied dataset to: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug_v2
Removed cache: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug_v2\labels\train.cache
Removed cache: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug_v2\labels\val.cache

Updated data.yaml:
path: C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/archive4_yolo_aug_v2
train: images/train
val: images/val
names:
  0: be_den
  1: mat_bo_phan
  2: mop_lom
  3: rach
  4: thung
  5: tray_son
  6: vo_kinh

Removed cache: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug_v2\labels\train.cache
Removed cache: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo_aug_v2\labels\val.cache

Updated data.yaml:
path: C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/archive4_yolo_aug_v2
train: images/train
val: images/val
names:
  0: be_den
  1: mat_bo_phan
  2: mop_l

In [18]:
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False

def find_label_for_image(img_path):
    parts = list(img_path.parts)
    # replace images with labels in path
    parts[parts.index("images")] = "labels"
    label_path = Path(*parts).with_suffix(".txt")
    return label_path

removed = 0

for split in ["train", "val"]:
    img_dir = DST / "images" / split
    
    for img_path in list(img_dir.glob("*.*")):
        label_path = find_label_for_image(img_path)
        
        try:
            with Image.open(img_path) as img:
                img.verify()
            with Image.open(img_path) as img:
                img.load()
        except Exception as e:
            print(f"Removing corrupt {split} image:", img_path.name, "->", e)
            img_path.unlink(missing_ok=True)
            label_path.unlink(missing_ok=True)
            removed += 1

print("Total corrupt image-label pairs removed:", removed)

Removing corrupt train image: 02012020_082351image833616.jpg -> image file is truncated (21 bytes not processed)
Removing corrupt train image: 04052020_152057image59498.jpg -> broken data stream when reading image file
Removing corrupt train image: 04052020_152107image748519.jpg -> image file is truncated (0 bytes not processed)
Removing corrupt val image: 04052020_152101image628633.jpg -> broken data stream when reading image file
Total corrupt image-label pairs removed: 4


In [19]:
def count_files(folder, pattern):
    return len(list(folder.glob(pattern)))

for split in ["train", "val"]:
    img_count = count_files(DST / "images" / split, "*.*")
    lbl_count = count_files(DST / "labels" / split, "*.txt")
    print(f"{split} images:", img_count)
    print(f"{split} labels :", lbl_count)

train images: 13185
train labels : 13185
val images: 470
val labels : 470


In [20]:
from collections import Counter

def count_class_instances(label_dir):
    counter = Counter()
    
    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) > 0:
                    class_id = parts[0]
                    counter[class_id] += 1
    
    return counter

before_counts = count_class_instances(DST / "labels" / "train")

print("Class instance counts before v2 targeted augmentation:")
for class_id in sorted(before_counts.keys(), key=int):
    print(class_id, ":", before_counts[class_id])

Class instance counts before v2 targeted augmentation:
0 : 1729
1 : 2247
2 : 5145
3 : 3969
4 : 1628
5 : 15977
6 : 1764


In [22]:
from PIL import ImageEnhance, ImageFilter
import numpy as np
from io import BytesIO
import random

random.seed(42)
np.random.seed(42)

def aug_brightness(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

def aug_contrast(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)

def aug_blur(img, radius):
    return img.filter(ImageFilter.GaussianBlur(radius))

def aug_noise(img, noise_level):
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-noise_level, noise_level + 1, arr.shape, dtype=np.int16)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def aug_color(img, factor):
    return ImageEnhance.Color(img).enhance(factor)

def aug_sharpness(img, factor):
    return ImageEnhance.Sharpness(img).enhance(factor)

def aug_jpeg_quality(img, quality=55):
    buffer = BytesIO()
    img.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")

In [24]:
WEAK_CLASS_IDS = {"0", "2", "3", "5"}

PREVIOUS_AUG_TAGS = (
    "_bright",
    "_dark",
    "_contrast",
    "_blur",
    "_noise",
    "_shadow"
)

def label_contains_weak_class(label_path):
    if not label_path.exists():
        return False
    
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) > 0 and parts[0] in WEAK_CLASS_IDS:
                return True
    
    return False

def is_original_stem(stem):
    return not stem.endswith(PREVIOUS_AUG_TAGS)

target_labels = []

for label_path in sorted((DST / "labels" / "train").glob("*.txt")):
    if not is_original_stem(label_path.stem):
        continue
    
    if label_contains_weak_class(label_path):
        target_labels.append(label_path)

print("Original train labels containing weak classes:", len(target_labels))
print("Sample:", [p.name for p in target_labels[:5]])

Original train labels containing weak classes: 1622
Sample: ['01012020_172204image853193.txt', '01012020_172204image891741.txt', '01022020_102246image365727.txt', '01022020_102909image891067.txt', '01022020_102910image826484.txt']


In [25]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

IMG_TRAIN = DST / "images" / "train"
LBL_TRAIN = DST / "labels" / "train"

def find_matching_image(stem):
    for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
        img_path = IMG_TRAIN / f"{stem}{ext}"
        if img_path.exists():
            return img_path
    return None

extra_aug_count = 0
missing_image_count = 0
broken_count = 0

for label_path in target_labels:
    stem = label_path.stem
    img_path = find_matching_image(stem)
    
    if img_path is None:
        missing_image_count += 1
        continue
    
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception as e:
        print("Skipping broken image:", img_path.name, "->", e)
        broken_count += 1
        continue
    
    suffix = img_path.suffix
    
    augmentations = [
        ("v2_bright2", aug_brightness(img, 1.35)),
        ("v2_dark2", aug_brightness(img, 0.65)),
        ("v2_contrast2", aug_contrast(img, 1.45)),
        ("v2_noise2", aug_noise(img, 18)),
        ("v2_blur2", aug_blur(img, 2.0)),
        ("v2_colorlow", aug_color(img, 0.65)),
        ("v2_sharp", aug_sharpness(img, 2.0)),
        ("v2_jpeg", aug_jpeg_quality(img, 55)),
    ]
    
    for tag, aug_img in augmentations:
        new_img_name = f"{stem}_{tag}{suffix}"
        new_lbl_name = f"{stem}_{tag}.txt"
        
        aug_img.save(IMG_TRAIN / new_img_name)
        shutil.copy2(label_path, LBL_TRAIN / new_lbl_name)
        extra_aug_count += 1

print("Extra targeted augmented images created:", extra_aug_count)
print("Missing matching images:", missing_image_count)
print("Broken images skipped:", broken_count)

Extra targeted augmented images created: 12976
Missing matching images: 0
Broken images skipped: 0


In [26]:
for split in ["train", "val"]:
    img_count = count_files(DST / "images" / split, "*.*")
    lbl_count = count_files(DST / "labels" / split, "*.txt")
    print(f"{split} images:", img_count)
    print(f"{split} labels :", lbl_count)

after_counts = count_class_instances(DST / "labels" / "train")

print("\nClass instance counts after v2 targeted augmentation:")
for class_id in sorted(after_counts.keys(), key=int):
    before = before_counts.get(class_id, 0)
    after = after_counts.get(class_id, 0)
    print(f"{class_id}: {before} -> {after}  increase: {after - before}")

train images: 26161
train labels : 26161
val images: 470
val labels : 470

Class instance counts after v2 targeted augmentation:
0: 1729 -> 3705  increase: 1976
1: 2247 -> 3807  increase: 1560
2: 5145 -> 11025  increase: 5880
3: 3969 -> 8505  increase: 4536
4: 1628 -> 3316  increase: 1688
5: 15977 -> 34209  increase: 18232
6: 1764 -> 2628  increase: 864
